In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from pathlib import Path
from itertools import groupby
from operator import itemgetter
from CustomFunctions import DetailedBalance, utils

In [9]:
####### load common directories FOR APPLIED CONFOCAL PCA
time_interval = 5 #sec/frame
whichpcs = [1,2]
basedir = Path('E:/Aaron/Combined_37C_Confocal_PCA_s5_LLS_Apply')
datadir = basedir.joinpath('Data_and_Figs')
FullFrame = pd.read_csv(datadir.joinpath('All_Data_with_CGPS_bins.csv'), index_col=0)
centers = pd.read_csv(datadir.joinpath('PC_bin_centers.csv'), index_col=0)
nbins = len(centers.iloc[:,0])
center = [9,10] # define the center of the cycle to calculate aer around
ttot = 3600
ntranslist = [1,2,3]
bsiter = 3000

In [7]:
### restrict data to RANDOM
treatments = ['Random']

savedir = basedir.joinpath('random')
if not savedir.exists():
    savedir.mkdir()

#restrict dataframe to only random experiments
TotalFrame = FullFrame[FullFrame.Treatment=='Random'].copy()

In [8]:
if __name__ ==  '__main__':
    ########### get raw transitions and pairs ###########
    rawtrans = DetailedBalance.get_raw_cgps_trajectories(
            TotalFrame, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            time_interval, #real time between datapoints
            savedir, #where to save the aggregated trajectories
            )


    ########### interpolate all transitions so that only individual transitions are made ###########
    transdf_sep = DetailedBalance.get_interpolated_cgps_trajectories(
            rawtrans, #pandas dataframe with all of the cgps binned data
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated trajectories
            )
    
    
    ############## get the counts of cells leaving 
    trans_rate_df_sep = DetailedBalance.aggregate_transition_counts(
            transdf_sep, #transdf_sep from get_interpolated_cgps_trajectories
            whichpcs, #which two PCs to use in the cgps [x,y]
            savedir, #where to save the aggregated counts
            nbins, #how many bins in the x and y cgps axes
            )
    
    for ntrans in ntranslist:
        ############## BOOTSTRAP MANY TRAJECTORIES ##########
        bstrans, bsint, bsframe_sep_full = DetailedBalance.get_bootstrapped_cgps_trajectories(
                rawtrans, #raw transition pairs from get_raw_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                time_interval, #real time between datapoints
                savedir, #where to save the aggregated counts
                nbins, #how many bins in the x and y cgps axes
                ttot, #set the total bootstrap time
                ntrans, #how many transitions to sample at each step
                bsiter, #number of times to bootstrap
                )


        ############# calculate average bootstrapped currents ###################
        bsfield_sep = DetailedBalance.get_avg_current_error(
                bsframe_sep_full, #transition rates in the cgps from get_bootstrapped_cgps_trajectories
                whichpcs, #which two PCs to use in the cgps [x,y]
                savedir, #where to save the aggregated counts
                nbins, #how many bins in the x and y cgps axes
                ntrans, #how many transitions to sample at each step
                )


Aggregated transitions
Finished interpolating trajectories
Total time observed in this CGPS was 494.9719140383019 minutes
Finished finding transition rates
Boostrapping trajectories with 1 transition samples for Random


100%|██████████| 3000/3000 [03:53<00:00, 12.83it/s]


Interpolating trajectories for Random


100%|██████████| 3000/3000 [00:05<00:00, 523.51it/s] 


Calculating bootstrapped CGPS transition rates for Random


...

Interpolating trajectories for Random


100%|██████████| 3000/3000 [00:05<00:00, 521.64it/s] 


Calculating bootstrapped CGPS transition rates for Random


100%|██████████| 3000/3000 [02:23<00:00, 20.87it/s]


Finished bootstrapping


In [11]:
################ get aer and cfs ##################
if __name__ ==  '__main__':
    for ntrans in ntranslist:
        ntrans_path = savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_bootstrapped_{ntrans}_transitions.csv')
        if ntrans_path.exists():
            bstrans = pd.read_csv(ntrans_path, index_col=0)
            xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]
            DetailedBalance.get_aer_cf(
                bstrans, #boostrapped transitions from get_bootstrapped_cgps_trajectories
                nbins, #how many bins in the x and y cgps axes
                xyscaling, #scaling of the bins in real units of whatever the CGPS axis parameters are
                center, #origin in [x bin,y bin]
                savedir, #where to save calculated aers and cfs
                whichpcs, #which two PCs to use in the cgps [x,y]
                ntrans,
                )

100%|██████████| 3000/3000 [00:14<00:00, 211.63it/s]


In [13]:
########## get RAW individual cell actual aer and cfs ###############

#get the area scaling in x and y based on the size of the bins in the cgps
xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

#open the raw transitions in case I didn't just generate them
rawtrans = pd.read_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_transitions_separated.csv'), index_col = 0)
#add a movie columns to separate dataframe on
rawtrans['Movie'] = [x.split('_frame')[0] for x in rawtrans.cell.to_list()]
#set the origin to the actual center
results = []
for i, cells in rawtrans.groupby('CellID'):
    movielist = sorted(cells.Movie.unique(),key = lambda x: int(x.split('-')[-3]))
    for m in movielist:
        curmov = cells[cells.Movie == m]
        curmov, runs = utils.get_consecutive_timepoints(curmov, 'frame',1)
        for r in runs:
            cell = curmov.iloc[r].reset_index(drop=True)
            results.append(DetailedBalance.get_area_enclosing_rate((
                cell,
                nbins,
                xyscaling,
                center,
                )))

#make a dataframe and save it
allaers = pd.concat(results, ignore_index=True)
justaers = allaers[['CellID','cell','Treatment','Movie','frame','aer','angular_velocity']].copy()
justaers.to_csv(savedir.joinpath(f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv'))


In [8]:
########## get INTERPOLATED individual cell actual aer and cfs ###############

#get the area scaling in x and y based on the size of the bins in the cgps
xyscaling = [centers[f'PC{whichpcs[0]}'].diff().mean(),centers[f'PC{whichpcs[1]}'].diff().mean()]

#open the raw transitions in case I didn't just generate them
interptrans = pd.read_csv(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_interpolated_transitions_separated.csv', index_col = 0)
#set the origin to the actual center
results = []
for i, cell in interptrans.groupby('CellID'):
    cell = cell.sort_values(['real_time','cumulative_time']).reset_index(drop = True)
    diff = cell.cumulative_time.diff()
    difflist = [0]
    difflist.extend(diff[diff<0].index.to_list())
    if difflist[-1] < len(cell):
        difflist.append(len(cell))
    #make a list of lists with the indices of consecutive time points
    runs = [list(range(difflist[x], difflist[x+1])) for x in range(len(difflist)-1)]
    for r in runs:
        cellrun = cell.iloc[r].reset_index(drop=True)
        results.append(DetailedBalance.get_area_enclosing_rate((
            cellrun,
            nbins,
            xyscaling,
            center,
            )))

#make a dataframe and save it
allaers = pd.concat(results, ignore_index=True)
allaers.to_csv(savedir + f'PC{whichpcs[0]}-PC{whichpcs[1]}_interpolated_transition_aer_cf.csv')


In [31]:
############ create gaps in the bootstrapped data similar to the real data ############
ntrans = 1
justaers = pd.read_csv(savedir + f'PC{whichpcs[0]}-PC{whichpcs[1]}_raw_transition_aer_cf.csv', index_col = 0)
realcelldf = TotalFrame.merge(justaers[['aer','cell']], on = 'cell', how = 'left')

########## measure gap frequency and duration
gap_prob_frame, gap_frame_num = DetailedBalance.get_gap_stats(
        realcelldf, #dataframe
        'CellID', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )
print(f'Average track gap probability for real data is {gap_prob_frame}')

#### get bs data with gaps
bsaers = pd.read_csv(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates.csv', index_col=0)
bs_with_gaps = DetailedBalance.bootstrap_gaps(
    bsaers, #dataframe with bootstrap iterations (doesn't actually need aer)
    0.035, #gap probability NOTE: this won't necessarily match the gap frequency in the output
    gap_frame_num, #distribution of gap lengths in numbers of frames
    )

### measure the gap probability in the newly gapped bootstrap data
#change real_time to just time
bs_gap_measure = bs_with_gaps.rename(columns = {'real_time':'time'})
#add dummy column
bs_gap_measure['aer'] = 0
bsgap_prob_frame, bsgap_frame_num = DetailedBalance.get_gap_stats(
        bs_gap_measure, #dataframe
        'iter', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )

print(f'Average track gap probability for the bootstrapped data is {bsgap_prob_frame}')

### save the gapped bootstrap data
bs_with_gaps.to_csv(savedir+f'PC{whichpcs[0]}-PC{whichpcs[1]}_{ntrans}_transition_Area_Enclosing_Rates_gaps.csv')


Average track gap probability for real data is 0.02716344180792462
Average track gap probability for the bootstrapped data is 0.026554295632110918


In [30]:
# realcelldf[~realcelldf.aer.isna()].CellID.value_counts().min()
longiters = bs_with_gaps.iter.value_counts()>realcelldf[~realcelldf.aer.isna()].CellID.value_counts().min()
longbs = bs_with_gaps[bs_with_gaps.iter.isin(longiters[longiters==True].index.to_list())]

longbs = longbs.rename(columns = {'real_time':'time'})
#add dummy column
longbs['aer'] = 0
bsgap_prob_frame, bsgap_frame_num = DetailedBalance.get_gap_stats(
        longbs, #dataframe
        'iter', #what is the identifier to group by as a str
        time_interval, #frame rate of the data
        )

print(f'Average track gap probability for the bootstrapped data is {bsgap_prob_frame}')


Average track gap probability for the bootstrapped data is 0.030505341809901353
